# PySpark DataFrame DSL — Part 1

## Topics Covered

1. `select()` — Selecting columns
2. `show()` — Displaying DataFrame data
3. `columns` — Getting column names
4. `dtypes` — Getting column names and data types

---

# 1. `select()` — Selecting Columns

## Definition

`select()` is a DataFrame DSL method used to select specific columns or column expressions from a DataFrame.

It is similar to the `SELECT` clause in SQL.

### Basic Syntax

```python
df.select("column_name")
```

For multiple columns:

```python
df.select("column1", "column2", "column3")
```

---

## Example DataFrame

We will use this DataFrame for the examples in this section.

```python
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("DataFrameDSLPart1") \
    .getOrCreate()

data = [
    ("C001", "Karthik", "Chennai", 25, 55000.50),
    ("C002", "Arun", "Mumbai", 28, 62000.75),
    ("C003", "Rahul", "Delhi", 26, 48000.25),
    ("C004", "Priya", "Bangalore", 30, 75000.00)
]

columns = [
    "customer_id",
    "name",
    "city",
    "age",
    "salary"
]

df = spark.createDataFrame(data, columns)

df.show()
```

---

## Example 1 — Select One Column

```python
df.select("name").show()
```

Output:

```text
+-------+
|   name|
+-------+
|Karthik|
|   Arun|
|  Rahul|
|  Priya|
+-------+
```

### What happened?

The original DataFrame contains:

```text
customer_id
name
city
age
salary
```

After:

```python
df.select("name")
```

the resulting DataFrame contains only:

```text
name
```

---

## Example 2 — Select Multiple Columns

```python
df.select(
    "name",
    "city"
).show()
```

Output:

```text
+-------+---------+
|   name|     city|
+-------+---------+
|Karthik|  Chennai|
|   Arun|   Mumbai|
|  Rahul|    Delhi|
|  Priya|Bangalore|
+-------+---------+
```

---

## Example 3 — Select Three Columns

```python
df.select(
    "name",
    "age",
    "salary"
).show()
```

This returns only:

```text
name
age
salary
```

---

## Example 4 — Select All Columns

You can use:

```python
df.select("*").show()
```

This selects every column.

It is equivalent to:

```python
df.show()
```

for displaying the complete DataFrame.

However, `select("*")` returns a DataFrame, while `show()` displays the DataFrame.

---

## Example 5 — Select Columns in a Different Order

You are not required to select columns in their original order.

```python
df.select(
    "salary",
    "name",
    "city"
).show()
```

The output order will be:

```text
salary
name
city
```

The original DataFrame remains unchanged.

---

## Example 6 — Store the Result in Another DataFrame

```python
df2 = df.select(
    "name",
    "city"
)

df2.show()
```

Now:

```text
df
 ↓
Original DataFrame

df2
 ↓
DataFrame containing name and city
```

Important:

```python
df
```

still contains all five columns.

---

## Example 7 — Select Using `col()`

Instead of:

```python
df.select("name", "salary").show()
```

you can use:

```python
from pyspark.sql.functions import col

df.select(
    col("name"),
    col("salary")
).show()
```

This becomes especially useful when creating column expressions.

---

## Example 8 — Select a Calculated Column

```python
df.select(
    col("name"),
    col("salary") * 12
).show()
```

This calculates annual salary.

For a meaningful column name:

```python
df.select(
    col("name"),
    (col("salary") * 12).alias("annual_salary")
).show()
```

---

## Example 9 — Select With Multiple Expressions

```python
df.select(
    col("name"),
    (col("age") + 1).alias("next_year_age"),
    (col("salary") * 12).alias("annual_salary")
).show()
```

Conceptually:

```text
name
age → age + 1 → next_year_age
salary → salary × 12 → annual_salary
```

---

## Example 10 — SQL Comparison

PySpark:

```python
df.select(
    "name",
    "city",
    "salary"
).show()
```

Equivalent SQL:

```sql
SELECT name, city, salary
FROM customers;
```

This is one of the reasons the DataFrame API is considered a DataFrame DSL.

---

## Important Points About `select()`

### 1. It returns a new DataFrame

```python
df2 = df.select("name")
```

It does not modify `df`.

### 2. It can select columns

```python
df.select("name", "city")
```

### 3. It can select expressions

```python
df.select(col("salary") * 12)
```

### 4. It can use aliases

```python
df.select(
    col("salary").alias("monthly_salary")
)
```

### Mental Model

```text
Original DataFrame
        |
        | select()
        ↓
Selected DataFrame
```

---

# 2. `show()` — Displaying DataFrame Data

## Definition

`show()` is a DataFrame method used to display rows from a DataFrame in a readable tabular format.

### Basic Syntax

```python
df.show()
```

By default, Spark displays the first **20 rows**.

---

## Example 1 — Basic `show()`

```python
df.show()
```

Output:

```text
+-----------+-------+---------+---+--------+
|customer_id|   name|     city|age|  salary|
+-----------+-------+---------+---+--------+
|       C001|Karthik|  Chennai| 25|55000.50|
|       C002|   Arun|   Mumbai| 28|62000.75|
|       C003|  Rahul|    Delhi| 26|48000.25|
|       C004|  Priya|Bangalore| 30|75000.00|
+-----------+-------+---------+---+--------+
```

---

## Example 2 — Display Only 2 Rows

```python
df.show(2)
```

This displays the first two rows.

```text
+-----------+-------+-------+---+--------+
|customer_id|   name|   city|age|  salary|
+-----------+-------+-------+---+--------+
|       C001|Karthik|Chennai| 25|55000.50|
|       C002|   Arun| Mumbai| 28|62000.75|
+-----------+-------+-------+---+--------+
```

---

## Example 3 — Display 3 Rows

```python
df.show(3)
```

This displays the first three rows.

---

## Example 4 — `show()` After `select()`

```python
df.select(
    "name",
    "city"
).show()
```

Here:

```text
select()
   ↓
Creates selected DataFrame
   ↓
show()
   ↓
Displays the result
```

---

## Example 5 — Avoid Truncating Long Strings

By default, Spark may truncate long string values.

You can use:

```python
df.show(truncate=False)
```

This tells Spark not to truncate string values.

Example:

```python
data = [
    ("C001", "Karthik", "Chennai - Tamil Nadu - India")
]

df_long = spark.createDataFrame(
    data,
    ["customer_id", "name", "location"]
)

df_long.show()
```

Some values may be shortened depending on their length.

Use:

```python
df_long.show(truncate=False)
```

to display the complete value.

---

## Example 6 — Specify Number of Rows and Truncation

```python
df.show(2, truncate=False)
```

Meaning:

```text
2
↓
Display first 2 rows

truncate=False
↓
Do not truncate string values
```

---

## Example 7 — Vertical Display

You can use:

```python
df.show(vertical=True)
```

This displays each row vertically.

Example concept:

```text
-RECORD 0----------------
 customer_id | C001
 name        | Karthik
 city        | Chennai
 age         | 25
 salary      | 55000.5
```

This can be useful when there are many columns.

---

## Example 8 — Vertical Display With a Limit

```python
df.show(2, vertical=True)
```

This displays the first two rows vertically.

---

## `show()` Important Point

`show()` is mainly used for **inspection and debugging**.

For example:

```python
df.show()
```

helps you understand the data.

But:

```python
df.show()
```

does not create a new DataFrame.

Compare:

```python
df2 = df.select("name")
```

This creates a new DataFrame.

Whereas:

```python
df.show()
```

only displays data.

---

## Common `show()` Patterns

```python
df.show()
```

```python
df.show(5)
```

```python
df.show(5, truncate=False)
```

```python
df.show(vertical=True)
```

```python
df.show(5, vertical=True)
```

---

# 3. `columns` — Getting Column Names

## Definition

`columns` is a DataFrame property that returns the names of all columns in the DataFrame.

### Basic Syntax

```python
df.columns
```

Example:

```python
print(df.columns)
```

Output:

```text
['customer_id', 'name', 'city', 'age', 'salary']
```

---

## Important: `columns` Is a Property

Notice:

```python
df.columns
```

not:

```python
df.columns()
```

There are no parentheses because `columns` is a property, not a method.

---

## Example 1 — Print All Column Names

```python
print(df.columns)
```

Output:

```text
['customer_id', 'name', 'city', 'age', 'salary']
```

---

## Example 2 — Get the First Column Name

Python lists are zero-indexed.

```python
print(df.columns[0])
```

Output:

```text
customer_id
```

---

## Example 3 — Get the Second Column

```python
print(df.columns[1])
```

Output:

```text
name
```

---

## Example 4 — Loop Through Column Names

```python
for column in df.columns:
    print(column)
```

Output:

```text
customer_id
name
city
age
salary
```

This becomes useful when dynamically processing columns.

---

## Example 5 — Count the Number of Columns

```python
print(len(df.columns))
```

Output:

```text
5
```

This is useful when checking the structure of a DataFrame.

---

## Example 6 — Check Whether a Column Exists

Because `df.columns` returns a Python list:

```python
if "salary" in df.columns:
    print("Salary column exists")
```

Output:

```text
Salary column exists
```

---

## Example 7 — Dynamically Select Columns

Suppose:

```python
required_columns = ["name", "city", "salary"]
```

You can select them dynamically:

```python
df2 = df.select(*required_columns)

df2.show()
```

### Why `*`?

The list:

```python
required_columns
```

contains:

```text
["name", "city", "salary"]
```

Using:

```python
*required_columns
```

unpacks the list into:

```python
df.select("name", "city", "salary")
```

This is very useful in reusable ETL code.

---

## Example 8 — Rename Columns Using `columns`

You can inspect the column names:

```python
print(df.columns)
```

Then perform operations based on them.

For example:

```python
for column in df.columns:
    print("Column:", column)
```

This is useful when building dynamic data-processing logic.

---

## Important Points About `columns`

`df.columns`:

- Returns column names
- Returns a Python list
- Can be indexed
- Can be iterated
- Can be used for dynamic column selection

### Mental Model

```text
DataFrame
    |
    ↓
df.columns
    |
    ↓
Python list
    |
    +-- customer_id
    +-- name
    +-- city
    +-- age
    +-- salary
```

---

# 4. `dtypes` — Getting Column Names and Data Types

## Definition

`dtypes` is a DataFrame property that returns a list containing:

```text
(column_name, data_type)
```

for every column.

### Basic Syntax

```python
df.dtypes
```

Example:

```python
print(df.dtypes)
```

Output:

```text
[
    ('customer_id', 'string'),
    ('name', 'string'),
    ('city', 'string'),
    ('age', 'bigint'),
    ('salary', 'double')
]
```

The exact numeric type can depend on how the DataFrame was created and inferred.

---

## Important: `dtypes` Is Also a Property

Use:

```python
df.dtypes
```

not:

```python
df.dtypes()
```

Because `dtypes` is a property.

---

## Example 1 — Display All Data Types

```python
print(df.dtypes)
```

---

## Example 2 — Loop Through `dtypes`

```python
for column, datatype in df.dtypes:
    print(column, datatype)
```

Output:

```text
customer_id string
name string
city string
age bigint
salary double
```

---

## Example 3 — Get the First Column's Data Type

```python
print(df.dtypes[0])
```

Output:

```text
('customer_id', 'string')
```

This is a tuple containing:

```text
(column_name, data_type)
```

---

## Example 4 — Get Only the Data Type

```python
print(df.dtypes[0][1])
```

Output:

```text
string
```

Explanation:

```text
df.dtypes
    ↓
List
    ↓
[0]
    ↓
First tuple
    ↓
[1]
    ↓
Data type
```

---

## Example 5 — Find the Data Type of a Specific Column

```python
for column, datatype in df.dtypes:
    if column == "salary":
        print(datatype)
```

Output:

```text
double
```

---

## Example 6 — Check Whether a Column Is Numeric

A simple approach:

```python
for column, datatype in df.dtypes:
    if datatype in ["int", "bigint", "double", "float"]:
        print(column, "is numeric")
```

Possible output:

```text
age is numeric
salary is numeric
```

For production-quality schema handling, Spark's type classes and `printSchema()` provide more robust type information.

---

## `dtypes` vs `printSchema()`

Both help you understand DataFrame types, but they present information differently.

### `dtypes`

```python
df.dtypes
```

Returns a Python list:

```text
[
    ('customer_id', 'string'),
    ('name', 'string'),
    ('city', 'string'),
    ('age', 'bigint'),
    ('salary', 'double')
]
```

### `printSchema()`

```python
df.printSchema()
```

Displays Spark's tree-style schema:

```text
root
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: long (nullable = true)
 |-- salary: double (nullable = true)
```

### Key Difference

```text
dtypes
  ↓
Python-friendly list

printSchema()
  ↓
Detailed Spark schema display
```

`printSchema()` also shows nullability and nested structure more clearly.

---

# Combining All Four Topics

These four operations are frequently used together while exploring a DataFrame.

## Example 1 — Inspect the DataFrame

```python
print(df.columns)
print(df.dtypes)

df.show()
```

This tells you:

```text
columns → What columns exist?
dtypes  → What are their data types?
show()  → What does the data look like?
```

---

## Example 2 — Select and Display

```python
df.select(
    "name",
    "city"
).show()
```

---

## Example 3 — Dynamically Select and Display

```python
required_columns = [
    "customer_id",
    "name",
    "salary"
]

df.select(*required_columns).show()
```

---

## Example 4 — Inspect Column Names and Types Together

```python
for column, datatype in df.dtypes:
    print(f"Column: {column}, Type: {datatype}")
```

---

# Practical Data Engineering Pattern

When you receive a new dataset, a common first inspection process is:

```python
df.show()
```

Then:

```python
df.printSchema()
```

Then:

```python
print(df.columns)
```

Then:

```python
print(df.dtypes)
```

Then select the columns you need:

```python
df.select(
    "customer_id",
    "name",
    "salary"
).show()
```

Conceptually:

```text
              New DataFrame
                    |
                    ↓
                show()
                    |
              Understand data
                    |
                    ↓
             printSchema()
                    |
              Understand schema
                    |
                    ↓
              columns / dtypes
                    |
           Inspect structure/types
                    |
                    ↓
                select()
                    |
           Choose required columns
```

---

# Quick Reference

| Operation | Type | Purpose |
|---|---|---|
| `select()` | Method | Select columns/expressions |
| `show()` | Method | Display rows |
| `columns` | Property | Get column names |
| `dtypes` | Property | Get column names and data types |

---

# Method vs Property

This distinction is important.

## Methods

Methods use parentheses:

```python
df.select(...)
df.show(...)
```

## Properties

Properties do not use parentheses:

```python
df.columns
df.dtypes
```

Mental model:

```text
METHOD
df.select(...)
df.show(...)

PROPERTY
df.columns
df.dtypes
```

---

# Final Mental Model

```text
                    DataFrame
                        |
        +---------------+---------------+
        |               |               |
        ↓               ↓               ↓
    columns          dtypes           show()
        |               |               |
  Column names     Data types       Actual rows
        |
        ↓
     select()
        |
        ↓
 Selected DataFrame
```

These four operations form the basic foundation for **DataFrame DSL Part 1**.

Once these are comfortable, the next important DataFrame DSL topic is filtering:

```python
df.filter(...)
```

and:

```python
df.where(...)
```
